# Lab 13A: Diffusion

This week's lab consists of **two short notebooks** on two of the main ideas of the lecture:

| Notebook | Question |
|---|---|
| **13A** Diffusion | How does a diffusion model turn noise into an image, and how is it trained? |
| **13B** Self-supervised learning with SimCLR | Can a network learn useful features from images **without labels**, and how many labels does that save? |

Both are meant to be quick. **Use the rest of the session for your project.**

**Before you start: Runtime → Change runtime type → GPU.** Each notebook trains for a few minutes.

**Theory ↔ code.** As in the previous labs, the tables marked **🔗 Theory ↔ code** show how the equations
from the lecture map to lines of PyTorch.

**Cells marked 🛠️ TASK contain gaps (`???`) that you must fill in.** Each comes with a test cell; don't
move on until the tests pass. Write down your ✍️ **Prediction** before running an experiment.

---
## This notebook

We build a small **DDPM** (denoising diffusion probabilistic model, Lecture 13, Section 4) and train it to
generate MNIST digits. The code follows **Lecture 13, Code 2**; you fill in the lines that are the
equations of the lecture.

| Part | You will… |
|---|---|
| 1. The forward process | add noise to images in one step, for any $t$ |
| 2. The denoiser | look at the U-Net and how it is told the time step $t$ |
| 3. Training | write the training objective: predict the noise |
| 4. Sampling | write one step of the reverse process and generate new digits |
| 5. Exam-style questions | check your understanding |

**Connection to Lab 9B.** The denoising autoencoder in Lab 9B learned to remove noise of *one* strength
in *one* step. A diffusion model learns to remove noise of *every* strength, and generates an image by
removing it a little at a time, in many steps.

## Setup

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

import torchvision

mnist = torchvision.datasets.MNIST('.', train=True, download=True)
X = mnist.data.float().unsqueeze(1) / 255 * 2 - 1      # (60000, 1, 28, 28), pixels in [-1, 1]
print(f'{len(X)} training images, pixel range [{X.min():.0f}, {X.max():.0f}]')


def show(x, title=None, nrow=8, titles=None):
    # x: (n, 1, 28, 28) in [-1, 1]
    x = x.detach().cpu().clamp(-1, 1)
    n = len(x)
    ncol = min(n, nrow)
    fig, axs = plt.subplots(math.ceil(n / ncol), ncol, figsize=(1.3 * ncol, 1.4 * math.ceil(n / ncol)), squeeze=False)
    for ax in axs.flat:
        ax.axis('off')
    for k in range(n):
        axs.flat[k].imshow(x[k, 0], cmap='gray', vmin=-1, vmax=1)
        if titles is not None:
            axs.flat[k].set_title(titles[k], fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()

---
# Part 1 — The forward process

The forward process destroys an image $\mathbf{x}_0$ in $T = 1000$ small steps. At step $t$ it shrinks the
image a little and adds a little Gaussian noise, with a variance $\beta_t$ that grows from $10^{-4}$ to $0.02$
(the **noise schedule**, Lecture 13, Section 4.2):

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I}\right)$$

Because every step adds independent Gaussian noise, all $t$ steps together are again one Gaussian, so we can
jump from $\mathbf{x}_0$ to $\mathbf{x}_t$ **in one step**. With $\alpha_t = 1 - \beta_t$ and
$\bar\alpha_t = \alpha_1 \alpha_2 \cdots \alpha_t$:

$$\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol\varepsilon, \qquad \boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

The schedule below is the `NoiseSchedule` of Code 2. (Python counts the steps from 0, so `t = 0` is the
lecture's $t = 1$ and `t = 999` is $t = T$.)

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T)       # beta_t
alphas = 1 - betas                          # alpha_t = 1 - beta_t
alpha_bar = torch.cumprod(alphas, dim=0)    # alpha_bar_t = alpha_1 * ... * alpha_t

plt.figure(figsize=(6, 3.5))
plt.plot(alpha_bar.sqrt(), label=r'$\sqrt{\bar\alpha_t}$  (weight of the image)')
plt.plot((1 - alpha_bar).sqrt(), label=r'$\sqrt{1-\bar\alpha_t}$  (weight of the noise)')
plt.xlabel('t'); plt.legend(); plt.title('How much image and how much noise is in $x_t$'); plt.show()

### 🛠️ TASK 1: Noise an image in one step

**🔗 Theory ↔ code**

| Lecture | Code |
|---|---|
| $\sqrt{\bar\alpha_t}$ for each image's own $t$ | `a` |
| $\sqrt{1-\bar\alpha_t}$ | `b` |
| $\boldsymbol\varepsilon$ | `noise` |
| $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon$ | `x_t = ...` |

In [ ]:
def q_sample(x0, t, noise):
    # x0: (B, 1, 28, 28) clean images, t: (B,) time steps, noise: like x0
    a = alpha_bar.to(x0.device)[t].sqrt().view(-1, 1, 1, 1)         # sqrt(alpha_bar_t), one per image
    b = (1 - alpha_bar.to(x0.device)[t]).sqrt().view(-1, 1, 1, 1)   # sqrt(1 - alpha_bar_t)
    x_t = ???                                                       # x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps
    return x_t

In [ ]:
# Test for Task 1
_x0, _eps = torch.randn(3, 1, 28, 28), torch.randn(3, 1, 28, 28)
_t = torch.tensor([0, 500, 999])
_xt = q_sample(_x0, _t, _eps)
assert _xt.shape == _x0.shape, 'x_t should have the same shape as x_0'
for _k in range(3):
    _ab = alpha_bar[_t[_k]]
    assert torch.allclose(_xt[_k], _ab.sqrt() * _x0[_k] + (1 - _ab).sqrt() * _eps[_k], atol=1e-5), f'wrong x_t for t = {_t[_k].item()}'
print('✅ Task 1 passed')

In [ ]:
ts = torch.tensor([0, 50, 100, 200, 300, 500, 700, 999])
x0 = X[7:8].repeat(len(ts), 1, 1, 1)
show(q_sample(x0, ts, torch.randn_like(x0)), 'One digit, noised to different time steps', titles=[f't = {t}' for t in ts.tolist()])

### Questions 1

1. At which $t$ can you no longer recognise the digit? What is $\mathbf{x}_T$ (`t = 999`), roughly?
2. Each step multiplies the image by $\sqrt{1-\beta_t} < 1$ before adding noise of variance $\beta_t$. Why shrink the image? (Hint:
   what happens to the variance of $\mathbf{x}_t$ over many steps if we only added noise?)
3. Why is it so important for **training** that $\mathbf{x}_t$ can be computed from $\mathbf{x}_0$ in one step?
4. Does the forward process have any learnable parameters?

---
# Part 2 — The denoiser

The network $\boldsymbol\varepsilon_\theta(\mathbf{x}_t, t)$ gets a noisy image and the time step, and
predicts the noise in it: its output has the same shape as the image (Section 4.5). It is a small **U-Net**
(as in Lab 9B: downsampling, upsampling and skip connections), built from the residual blocks of
Code 2.

**How does the network know $t$?** The integer $t$ is turned into a vector with the **sinusoidal
embedding** you implemented for positions in 12A, passed through a small MLP, and *added* to the feature maps
of every residual block. The same network can then behave differently at every noise level. The
model is given; read it, and answer the questions below.

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    # Lecture 13, Code 2: the integer t -> a d-dimensional vector (the position embedding of 12A, with t as the position)
    def __init__(self, d):
        super().__init__()
        self.d = d

    def forward(self, t):
        half = self.d // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    # Lecture 13, Code 2: a residual block that adds the time embedding to every position
    def __init__(self, channels, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, channels))
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]     # the time step enters here
        h = self.conv2(F.silu(self.norm2(h)))
        return x + h


class UNet(nn.Module):
    def __init__(self, c=32, time_dim=128):
        super().__init__()
        self.time_embed = nn.Sequential(SinusoidalTimeEmbed(time_dim), nn.Linear(time_dim, time_dim), nn.SiLU(),
                                        nn.Linear(time_dim, time_dim))
        self.inp = nn.Conv2d(1, c, 3, padding=1)
        self.down1, self.res1 = nn.Conv2d(c, 2 * c, 4, 2, 1), ResBlock(2 * c, time_dim)            # 28 -> 14
        self.down2, self.res2 = nn.Conv2d(2 * c, 4 * c, 4, 2, 1), ResBlock(4 * c, time_dim)        # 14 -> 7
        self.mid = ResBlock(4 * c, time_dim)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 4, 2, 1), nn.Conv2d(4 * c, 2 * c, 3, padding=1)   # 7 -> 14
        self.res3 = ResBlock(2 * c, time_dim)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 4, 2, 1), nn.Conv2d(2 * c, c, 3, padding=1)           # 14 -> 28
        self.res4 = ResBlock(c, time_dim)
        self.out = nn.Conv2d(c, 1, 3, padding=1)

    def forward(self, x, t):
        e = self.time_embed(t)
        h0 = self.inp(x)                                       # (B, c, 28, 28)
        h1 = self.res1(self.down1(h0), e)                      # (B, 2c, 14, 14)
        h2 = self.res2(self.down2(h1), e)                      # (B, 4c, 7, 7)
        h = self.mid(h2, e)
        h = self.res3(self.dec2(torch.cat([self.up2(h), h1], 1)), e)    # skip connection from h1
        h = self.res4(self.dec1(torch.cat([self.up1(h), h0], 1)), e)    # skip connection from h0
        return self.out(h)                                     # predicted noise: (B, 1, 28, 28)


_m = UNet()
print('Output shape:', tuple(_m(torch.randn(2, 1, 28, 28), torch.tensor([3, 700])).shape))
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')

### Questions 2

1. Why must the output have the same shape as the input?
2. Why does the network need to know $t$? What would go wrong if it had to guess how much noise there is?

---
# Part 3 — Training: predict the noise

The training objective of DDPM (Section 4.3) is remarkably simple: take a clean image, noise it to a
random time step, and ask the network to predict **the noise that was added**:

$$\mathcal{L} = \mathbb{E}_{t,\, \mathbf{x}_0,\, \boldsymbol\varepsilon}\left[\left\lVert \boldsymbol\varepsilon - \boldsymbol\varepsilon_\theta\big(\underbrace{\sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon}_{\mathbf{x}_t},\; t\big)\right\rVert^2\right]$$

This is Algorithm 1 in the lecture: sample $\mathbf{x}_0$, $t$ and $\boldsymbol\varepsilon$; compute $\mathbf{x}_t$;
predict the noise; take a gradient step on the squared error.

### 🛠️ TASK 2: The training loss

**🔗 Theory ↔ code**

| Lecture (Algorithm 1) | Code |
|---|---|
| sample $t$ uniformly | `t = torch.randint(0, T, ...)` |
| sample $\boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ | `noise = torch.randn_like(x0)` |
| $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon$ | your `q_sample` from Task 1 |
| $\hat{\boldsymbol\varepsilon} = \boldsymbol\varepsilon_\theta(\mathbf{x}_t, t)$ | `model(x_t, t)` |
| $\lVert \boldsymbol\varepsilon - \hat{\boldsymbol\varepsilon} \rVert^2$ | `F.mse_loss(noise_pred, noise)` |

In [ ]:
def diffusion_loss(model, x0):
    t = torch.randint(0, T, (x0.shape[0],), device=x0.device)   # 1. a random time step for every image
    noise = torch.randn_like(x0)                                # 2. random noise
    x_t = ???                                                   # 3. the noisy images
    noise_pred = ???                                            # 4. the network predicts the noise
    return F.mse_loss(noise_pred, noise)                        # 5. squared error

In [ ]:
# Test for Task 2: a "network" that always predicts zero noise has a loss of about 1 (the variance of the noise)
class _Zero(nn.Module):
    def forward(self, x, t):
        return torch.zeros_like(x)
_l = diffusion_loss(_Zero(), X[:512])
assert abs(_l.item() - 1) < 0.05, f'predicting zero noise should give a loss of about 1; got {_l.item():.3f}'
# and the network must get the noisy image and the time step
class _Spy(nn.Module):
    def forward(self, x, t):
        assert t.shape == (x.shape[0],), 'the network should get one time step per image'
        assert not torch.allclose(x, X[:8]), 'the network should get the NOISY images x_t, not x_0'
        return x * 0
diffusion_loss(_Spy(), X[:8])
print('✅ Task 2 passed')

Now we train. Every step uses a fresh random $t$ and fresh noise for each image, so the network learns all
noise levels at once. This takes about 3 minutes on a GPU.

✍️ **Prediction:** the loss starts at about 1. Why? Where do you think it ends?

In [ ]:
set_seed(0)
model = UNet().to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=2e-3)
steps = 3000
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, 2e-3, total_steps=steps)
X_dev = X.to(device)
losses = []
t0 = time.time()
for step in range(steps):
    x0 = X_dev[torch.randint(0, len(X), (128,), device=device)]
    loss = diffusion_loss(model, x0)
    optimiser.zero_grad(); loss.backward(); optimiser.step(); schedule.step()
    losses.append(loss.item())
    if step % 500 == 0:
        print(f'step {step}: loss {loss.item():.3f}  ({time.time() - t0:.0f} s)')

plt.figure(figsize=(6, 3))
plt.plot(np.convolve(losses, np.ones(50) / 50, mode='valid'))
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss (moving average)'); plt.show()

### Questions 3

1. Why does the loss start at about 1?
2. Why can this be called a *denoising* objective? How is it related to the denoising autoencoder of Lab 9B, and what is different?

---
# Part 4 — Sampling: from noise to digits

To generate an image we run the process backwards (Algorithm 2): start from pure noise
$\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and, for $t = T, \ldots, 1$, remove a little of the predicted noise.
One reverse step is

$$\mathbf{x}_{t-1} = \underbrace{\frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat{\boldsymbol\varepsilon}\right)}_{\text{mean } \boldsymbol\mu_\theta(\mathbf{x}_t,\, t)} + \sqrt{\beta_t}\,\mathbf{z}, \qquad \hat{\boldsymbol\varepsilon} = \boldsymbol\varepsilon_\theta(\mathbf{x}_t, t),\quad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

(no $\mathbf{z}$ in the very last step). In words: subtract a small, scaled part of the predicted noise, undo the shrinking
of the forward step, and add back a little fresh noise. The lecture's Code 2 uses this formula; we take it as given
(deriving it is beyond the course).

We also look at the model's current guess of the **clean** image, by solving the forward formula for $\mathbf{x}_0$:
$\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1-\bar\alpha_t}\,\hat{\boldsymbol\varepsilon}) / \sqrt{\bar\alpha_t}$.

### 🛠️ TASK 3: One reverse step

**🔗 Theory ↔ code**

| Lecture | Code |
|---|---|
| $\mathbf{x}_t$, $\hat{\boldsymbol\varepsilon}$ | `x`, `noise_pred` |
| $\beta_t$, $\alpha_t$, $\bar\alpha_t$ | `beta`, `alpha`, `ab` |
| $\boldsymbol\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat{\boldsymbol\varepsilon}\right)$ | `mean = ...` |
| $+\sqrt{\beta_t}\,\mathbf{z}$ | `beta.sqrt() * torch.randn_like(x)` (given) |

In [ ]:
def reverse_step(x, noise_pred, t):
    beta, alpha, ab = betas[t].to(x.device), alphas[t].to(x.device), alpha_bar[t].to(x.device)
    mean = ???                                                  # mu = (x_t - beta_t / sqrt(1 - alpha_bar_t) * eps_hat) / sqrt(alpha_t)
    if t > 0:
        return mean + beta.sqrt() * torch.randn_like(x)         # add a little fresh noise
    return mean                                                 # last step: no noise


@torch.no_grad()
def sample(model, n=32, snapshots=(999, 800, 600, 400, 200, 100, 50, 0)):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)               # x_T: pure noise
    history = []
    for t in reversed(range(T)):
        noise_pred = model(x, torch.full((n,), t, device=device))
        if t in snapshots:                                     # keep x_t and the guess of x_0 for plotting
            ab = alpha_bar[t].to(device)
            history.append((t, x[0].clone(), ((x[0] - (1 - ab).sqrt() * noise_pred[0]) / ab.sqrt()).clamp(-1, 1)))
        x = reverse_step(x, noise_pred, t)
    return x.clamp(-1, 1), history

In [ ]:
# Test for Task 3
_x, _e = torch.randn(2, 1, 28, 28), torch.randn(2, 1, 28, 28)
_out = reverse_step(_x, _e, 0)                    # t = 0: no fresh noise, so the result is exactly the mean
_expected = (_x - betas[0] / (1 - alpha_bar[0]).sqrt() * _e) / alphas[0].sqrt()
assert torch.allclose(_out, _expected, atol=1e-5), 'the mean is not (x - beta / sqrt(1 - alpha_bar) * eps_hat) / sqrt(alpha)'
print('✅ Task 3 passed')

✍️ **Prediction:** how many times is the network evaluated to generate one batch of digits?

In [ ]:
set_seed(1)
t0 = time.time()
samples, history = sample(model)
print(f'Sampling took {time.time() - t0:.0f} s')
show(samples, 'Generated digits')

show(torch.stack([h[1] for h in history]).clamp(-1, 1), 'One sample during the reverse process: x_t',
     titles=[f't = {h[0]}' for h in history])
show(torch.stack([h[2] for h in history]), "The model's guess of the clean image at that step",
     titles=[f't = {h[0]}' for h in history])

### Questions 4

1. How many network evaluations did it take to generate the digits? How many would a VAE decoder or a GAN generator need?
   What is the main practical disadvantage of diffusion models?
2. Look at the model's guess of the clean image along the way. What does it look like early (large $t$) and late (small
   $t$)? What is decided first: the rough shape or the details?
3. Why does sampling add fresh noise in every step except the last?

---
# Part 5 — Exam-style questions

**Q1.** In the forward process of a diffusion model, $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\varepsilon$. What does this formula make possible?

* (a) Generating an image in a single step.
* (b) Computing the noisy image at any time step directly from the clean image, without simulating the steps in between.
* (c) Learning the noise schedule.
* (d) Removing the need for a neural network.

**Q2.** What is the denoising network of a DDPM trained to predict?

* (a) The class of the image.
* (b) The noise that was added to the image.
* (c) The time step $t$.
* (d) The value of $\bar\alpha_t$.

**Q3.** How is the time step $t$ usually given to the U-Net of a diffusion model?

* (a) As an extra image channel filled with the value $t$.
* (b) It is not given; the network estimates it.
* (c) As a sinusoidal embedding, passed through an MLP and added to the feature maps of the residual blocks.
* (d) By using a different network for every $t$.

**Q4.** What is the main practical disadvantage of diffusion models compared with GANs?

* (a) Their training is unstable.
* (b) They suffer from mode collapse.
* (c) Generating an image requires many (e.g. 1,000) evaluations of the network.
* (d) They can only generate grey-scale images.